# 3.3 Search on a budget: where the trials go, and how long each one trains

At the end of this notebook you can:

- say how random search and TPE choose their next trial, and read what TPE buys from
  **where its trials go**, not from its best score;
- say how successive halving spends a budget (many trials short, only the best of each round
  longer), read what it saves, and name its pitfall: a trial that starts slowly is stopped
  before it would have won.

**Why it matters for your study.** A study spends its budget (`MINUTES` in `study.py`) in two
places: on which configurations it tries, the job of Optuna's **sampler**, which picks the
values of the next trial; and on how long it trains each one, the job of a **pruner**, which
stops a trial before its last epoch. Lesson 2's study used both without opening them; this
notebook opens them, on one stored search. The slides carry grid search, a population
(NSGA-II), how a space grows with its axes, the learning-rate scheduler and the Pareto front.

## 3.3.1 One search, stored

**The data.** Food-101 photos of 101 dishes, each turned into one vector by the frozen
`dinov2-large` encoder (the vectors of 3.1): 150 photos per class, 13,635 to train on and
1,515 held out for validation.

**One trial** trains a head on those vectors with AdamW (Adam with weight decay) for 20 epochs and records its
validation accuracy after every epoch: its whole **curve**. The objective, the number the
search maximises, is the validation accuracy after the last epoch. The search space has seven
axes:

| axis | values |
|---|---|
| `lr` | 1e-5 to 1e-1, log scale |
| `width` | 0 (a linear head) or 64, 128, 256, 512, 1024 hidden units |
| `depth` | 1 or 2 hidden layers (when `width` > 0) |
| `dropout` | 0 to 0.5 |
| `weight_decay` | 1e-6 to 1e-1, log scale: how hard AdamW pulls every weight towards zero |
| `batch` | 64, 128, 256 or 512 rows per update |
| `smoothing` | 0 to 0.2: label smoothing, the target gives this share of its probability to the other classes |

**Two searches** of 60 trials each ran on this space, one with Optuna's `RandomSampler` and one
with its `TPESampler`, both with sampler seed 0 and the same torch seed for every trial.
`search_budget.py`, next to this notebook, ran them once and stored every trial's values,
curve and seconds in `results/search_budget.json`; this notebook only reads that file. To run
them again, from the repository root: `uv run python notebooks/3_scaling_and_search/search_budget.py`.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd

optuna.logging.set_verbosity(optuna.logging.WARNING)

STORED = json.loads(Path("results/search_budget.json").read_text(encoding="utf-8"))
EPOCHS = STORED["epochs"]
SAMPLERS = ("random", "tpe")
COLOR = {"random": "#E69F00", "tpe": "#0072B2"}  # Okabe-Ito orange and blue
NAME = {"random": "random", "tpe": "TPE"}


def trials_of(sampler: str) -> pd.DataFrame:
    # One row per trial: its values, its accuracy after epoch 1 and 20, its seconds.
    return pd.DataFrame(
        [
            {
                "trial": r["number"],
                **r["params"],
                "epoch 1": r["curve"][0],
                "epoch 20": r["curve"][-1],
                "seconds": r["seconds"],
            }
            for r in STORED[sampler]
        ]
    ).set_index("trial")


TRIALS = {s: trials_of(s) for s in SAMPLERS}
CURVES = {s: np.array([r["curve"] for r in STORED[s]]) for s in SAMPLERS}  # (60, 20)

print(
    f"{STORED['train']:,} train and {STORED['valid']:,} validation vectors, {EPOCHS} epochs per trial"
)
for s in SAMPLERS:
    t = TRIALS[s]
    print(
        f"{s:6s}: {len(t)} trials, {t['seconds'].min():.1f} to {t['seconds'].max():.1f} s per trial, "
        f"{t['seconds'].sum():.0f} s in all"
    )
display(
    TRIALS["random"]
    .head(10)
    .style.format(
        {
            "lr": "{:.1e}",
            "weight_decay": "{:.0e}",
            "dropout": "{:.2f}",
            "smoothing": "{:.2f}",
            "epoch 1": "{:.3f}",
            "epoch 20": "{:.3f}",
            "seconds": "{:.1f}",
        }
    )
    .set_caption(
        "the first ten trials of the random search; validation accuracy after epoch 1 and 20"
    )
)

Every trial is one row like these, plus its curve of 20 accuracies. A linear head at batch 256
(trial 3) trains in under a second and reaches 0.888; 1,024 units at batch 128 (trials 8 and 9)
take five seconds and end at 0.842. Trial 7, at `lr` 6.5e-2, ends at 0.684: some regions of
this space are bad, and a search that keeps drawing there wastes its trials.

## 3.3.2 How random and TPE choose the next trial

**Random** draws every axis afresh from its range (the learning rate uniformly on its log
scale) and remembers nothing. Trial 50 is drawn exactly as trial 1 was, even after 49 trials
showed that a learning rate above 1e-2 ends far below the top.

**TPE** (Tree-structured Parzen Estimator) starts with 10 random trials. From then on, before
every trial it does four things:

1. **Split.** It sorts the trials so far by score. The best tenth, rounded up, is the
   **good** group, the others the **rest**. After 10 trials the good group is 1 trial; after
   40 trials it is 4.
2. **Model each group.** On a continuous axis such as `lr` it puts a bell on every trial's
   value, wide while there are few trials; on an axis with a few values, such as `batch`, a
   weight on each value. Added up, that gives two distributions: `l`, high where the good trials were,
   and `g`, high where the rest were.
3. **Draw candidates.** It draws 24 candidates from `l`.
4. **Pick.** It runs the candidate where `l` is high compared with `g`: a region that gave
   good trials and few bad ones.

So random keeps spending trials on regions it already saw fail (a trial **fails** here when it
ends below 0.8, far under the 0.89 the good trials reach); TPE draws more where the good trials
came from. The picture shows it on one axis:

<svg width="720" height="290" viewBox="0 0 720 290" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="13">
<rect width="720" height="290" fill="white"/>
<defs><marker id="tpe-arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#444"/></marker></defs>
<text x="10" y="24" font-weight="bold" fill="#222">TPE before one trial: the trials so far, split into the best tenth (good) and the rest</text>
<text x="14" y="66" fill="#009E73">good</text>
<text x="14" y="96" fill="#999999">rest</text>
<circle cx="294.8" cy="62" r="6" fill="#009E73"/><circle cx="99.0" cy="92" r="6" fill="#999999"/><circle cx="142.5" cy="92" r="6" fill="#999999"/><circle cx="193.2" cy="92" r="6" fill="#999999"/><circle cx="229.5" cy="92" r="6" fill="#999999"/><circle cx="367.2" cy="92" r="6" fill="#999999"/><circle cx="425.2" cy="92" r="6" fill="#999999"/><circle cx="476.0" cy="92" r="6" fill="#999999"/><circle cx="563.0" cy="92" r="6" fill="#999999"/><circle cx="606.5" cy="92" r="6" fill="#999999"/>
<polyline points="70.0,223.8 73.6,223.8 77.2,223.8 80.9,223.8 84.5,223.8 88.1,223.8 91.8,223.8 95.4,223.8 99.0,223.8 102.6,223.8 106.2,223.8 109.9,223.8 113.5,223.8 117.1,223.8 120.7,223.8 124.4,223.8 128.0,223.8 131.6,223.7 135.3,223.7 138.9,223.7 142.5,223.7 146.1,223.6 149.7,223.6 153.4,223.5 157.0,223.4 160.6,223.3 164.3,223.2 167.9,223.0 171.5,222.7 175.1,222.4 178.8,222.0 182.4,221.5 186.0,220.9 189.6,220.1 193.2,219.2 196.9,218.1 200.5,216.8 204.1,215.2 207.8,213.3 211.4,211.2 215.0,208.7 218.6,205.9 222.2,202.7 225.9,199.2 229.5,195.2 233.1,191.0 236.8,186.4 240.4,181.4 244.0,176.3 247.6,170.9 251.2,165.3 254.9,159.7 258.5,154.1 262.1,148.6 265.8,143.4 269.4,138.4 273.0,133.9 276.6,129.8 280.2,126.4 283.9,123.7 287.5,121.6 291.1,120.4 294.8,120.0 298.4,120.4 302.0,121.6 305.6,123.7 309.3,126.4 312.9,129.8 316.5,133.9 320.1,138.4 323.8,143.4 327.4,148.6 331.0,154.1 334.6,159.7 338.2,165.3 341.9,170.9 345.5,176.3 349.1,181.4 352.8,186.4 356.4,191.0 360.0,195.2 363.6,199.2 367.3,202.7 370.9,205.9 374.5,208.7 378.1,211.2 381.8,213.3 385.4,215.2 389.0,216.8 392.6,218.1 396.2,219.2 399.9,220.1 403.5,220.9 407.1,221.5 410.8,222.0 414.4,222.4 418.0,222.7 421.6,223.0 425.2,223.2 428.9,223.3 432.5,223.4 436.1,223.5 439.8,223.6 443.4,223.6 447.0,223.7 450.6,223.7 454.3,223.7 457.9,223.7 461.5,223.8 465.1,223.8 468.8,223.8 472.4,223.8 476.0,223.8 479.6,223.8 483.2,223.8 486.9,223.8 490.5,223.8 494.1,223.8 497.8,223.8 501.4,223.8 505.0,223.8 508.6,223.8 512.2,223.8 515.9,223.8 519.5,223.8 523.1,223.8 526.8,223.8 530.4,223.8 534.0,223.8 537.6,223.8 541.2,223.8 544.9,223.8 548.5,223.8 552.1,223.8 555.8,223.8 559.4,223.8 563.0,223.8 566.6,223.8 570.2,223.8 573.9,223.8 577.5,223.8 581.1,223.8 584.8,223.8 588.4,223.8 592.0,223.8 595.6,223.8 599.2,223.8 602.9,223.8 606.5,223.8 610.1,223.8 613.8,223.8 617.4,223.8 621.0,223.8 624.6,223.8 628.2,223.8 631.9,223.8 635.5,223.8 639.1,223.8 642.8,223.8 646.4,223.8 650.0,223.8" fill="none" stroke="#009E73" stroke-width="2.5"/>
<polyline points="70.0,212.4 73.6,211.4 77.2,210.4 80.9,209.4 84.5,208.5 88.1,207.5 91.8,206.6 95.4,205.7 99.0,204.9 102.6,204.1 106.2,203.4 109.9,202.8 113.5,202.2 117.1,201.6 120.7,201.1 124.4,200.7 128.0,200.3 131.6,200.0 135.3,199.8 138.9,199.5 142.5,199.3 146.1,199.1 149.7,199.0 153.4,198.8 157.0,198.7 160.6,198.6 164.3,198.5 167.9,198.4 171.5,198.4 175.1,198.3 178.8,198.3 182.4,198.3 186.0,198.3 189.6,198.3 193.2,198.4 196.9,198.6 200.5,198.8 204.1,199.1 207.8,199.5 211.4,199.9 215.0,200.4 218.6,201.0 222.2,201.7 225.9,202.4 229.5,203.2 233.1,204.1 236.8,205.0 240.4,206.0 244.0,207.0 247.6,208.1 251.2,209.1 254.9,210.1 258.5,211.1 262.1,212.1 265.8,213.1 269.4,213.9 273.0,214.7 276.6,215.5 280.2,216.1 283.9,216.7 287.5,217.1 291.1,217.4 294.8,217.7 298.4,217.8 302.0,217.8 305.6,217.7 309.3,217.5 312.9,217.3 316.5,216.9 320.1,216.4 323.8,215.9 327.4,215.3 331.0,214.7 334.6,214.0 338.2,213.3 341.9,212.6 345.5,211.8 349.1,211.1 352.8,210.4 356.4,209.7 360.0,209.0 363.6,208.3 367.3,207.7 370.9,207.2 374.5,206.6 378.1,206.1 381.8,205.7 385.4,205.3 389.0,204.9 392.6,204.5 396.2,204.2 399.9,203.9 403.5,203.7 407.1,203.5 410.8,203.3 414.4,203.1 418.0,203.0 421.6,202.9 425.2,202.8 428.9,202.7 432.5,202.7 436.1,202.7 439.8,202.8 443.4,202.8 447.0,203.0 450.6,203.1 454.3,203.4 457.9,203.6 461.5,203.9 465.1,204.2 468.8,204.6 472.4,205.0 476.0,205.4 479.6,205.8 483.2,206.3 486.9,206.7 490.5,207.1 494.1,207.5 497.8,207.9 501.4,208.2 505.0,208.5 508.6,208.7 512.2,208.8 515.9,208.9 519.5,208.8 523.1,208.7 526.8,208.6 530.4,208.3 534.0,208.0 537.6,207.7 541.2,207.2 544.9,206.8 548.5,206.3 552.1,205.9 555.8,205.4 559.4,205.0 563.0,204.6 566.6,204.2 570.2,203.9 573.9,203.7 577.5,203.5 581.1,203.4 584.8,203.5 588.4,203.6 592.0,203.8 595.6,204.1 599.2,204.6 602.9,205.1 606.5,205.7 610.1,206.4 613.8,207.1 617.4,207.9 621.0,208.8 624.6,209.7 628.2,210.6 631.9,211.5 635.5,212.5 639.1,213.4 642.8,214.4 646.4,215.3 650.0,216.1" fill="none" stroke="#999999" stroke-width="2"/>
<line x1="70" y1="186.0" x2="650" y2="186.0" stroke="#444" stroke-width="1.5" stroke-dasharray="5 4"/>
<text x="646" y="180.0" text-anchor="end" font-size="11" fill="#555">random: the same everywhere, for every trial</text>
<text x="128.0" y="116" font-size="12" fill="#009E73">l: where the good trials were</text>
<text x="461.5" y="116" font-size="12" fill="#999999">g: where the rest were</text>
<line x1="298.4" y1="108" x2="298.4" y2="228" stroke="#444" stroke-width="1.5" stroke-dasharray="3 3" marker-end="url(#tpe-arrow)"/>
<text x="306.4" y="150" font-size="12" fill="#222">next trial: l high, g low</text>
<line x1="70" y1="230" x2="650" y2="230" stroke="#444"/>
<line x1="70.0" y1="230" x2="70.0" y2="235" stroke="#444"/><text x="70.0" y="249" text-anchor="middle" font-size="11" fill="#222">1e-5</text><line x1="215.0" y1="230" x2="215.0" y2="235" stroke="#444"/><text x="215.0" y="249" text-anchor="middle" font-size="11" fill="#222">1e-4</text><line x1="360.0" y1="230" x2="360.0" y2="235" stroke="#444"/><text x="360.0" y="249" text-anchor="middle" font-size="11" fill="#222">1e-3</text><line x1="505.0" y1="230" x2="505.0" y2="235" stroke="#444"/><text x="505.0" y="249" text-anchor="middle" font-size="11" fill="#222">1e-2</text><line x1="650.0" y1="230" x2="650.0" y2="235" stroke="#444"/><text x="650.0" y="249" text-anchor="middle" font-size="11" fill="#222">1e-1</text>
<text x="360" y="270" text-anchor="middle" fill="#222">lr (log scale)</text>
</svg>

With the same seed, TPE's 10 random trials are random's first 10 trials. So the two searches
are identical up to trial 9 and part at trial 10, the first trial TPE picks from its model.

**What TPE knows at trial 10.** Its good group is one trial: the best of trials 0 to 9.

**Expect.** Write down: what will TPE's trials 10 to 19 have in common with that trial, and
random's?

In [ ]:
def head(row: pd.Series) -> str:
    width, depth = int(row["width"]), int(row["depth"])
    return "linear" if width == 0 else f"{width} x {depth}"


first = TRIALS["tpe"].iloc[:10]
best = first["epoch 20"].idxmax()
print(
    f"best of trials 0 to 9: trial {best}, {head(first.loc[best])}, batch "
    f"{int(first.loc[best, 'batch'])}, lr {first.loc[best, 'lr']:.1e}, ends at {first.loc[best, 'epoch 20']:.3f}\n"
)
print(f"{'trial':>5s} | {'random':^28s} | {'TPE':^28s}")
for n in range(10, 20):
    cells = []
    for s in SAMPLERS:
        row = TRIALS[s].loc[n]
        cells.append(
            f"{head(row):>10s}  batch {int(row['batch']):3d}  lr {row['lr']:.1e}"
        )
    print(f"{n:5d} | {cells[0]} | {cells[1]}")
print()
for s in SAMPLERS:
    block = TRIALS[s].iloc[10:20]
    print(
        f"{s:6s} trials 10 to 19: batch 512 in {(block['batch'] == 512).sum()}, "
        f"depth 2 in {(block['depth'] == 2).sum()}, width 256 in {(block['width'] == 256).sum()}"
    )

**Reading.** The good group at trial 10 is trial 4: two hidden layers of 256 at batch 512 and
`lr` 3.0e-4. All ten of TPE's next trials use batch 512, nine of them two hidden layers, six of
them width 256; random's next ten use batch 512 three times. TPE's learning rates spread around
trial 4's 3.0e-4, from 5.8e-5 to 6.9e-3: with one or two good trials the bells are wide. TPE
draws near the good trial, on every axis.

## 3.3.3 Random against TPE

**Best-so-far.** After every trial, the best score of all trials up to then. It can only go
up, and it is how a search is usually compared: how high, and after how many trials.

**Question.** Does TPE reach a better head than random within 60 trials, or reach it sooner?

**Expect.** Random keeps drawing where earlier trials failed; TPE draws less there. What
would the space have to look like for random to end as high anyway?

In [ ]:
fig, a = plt.subplots(figsize=(8, 3.8))
x = np.arange(60)  # trial numbers
for s in SAMPLERS:
    final = TRIALS[s]["epoch 20"].to_numpy()
    a.plot(x, final, "o", ms=3, color=COLOR[s], alpha=0.35)
    a.step(
        x,
        np.maximum.accumulate(final),
        where="post",
        color=COLOR[s],
        lw=2,
        ls="--" if s == "random" else "-",
        label=f"{NAME[s]}: best-so-far",
    )
a.axvline(9.5, color="0.5", lw=1, ls=":")
a.text(10, 0.805, "from trial 10 on, TPE picks from its model", fontsize=8, color="0.3")
a.set_ylim(0.8, 0.9)
a.set_xlabel("trial number")
a.set_ylabel("validation accuracy after epoch 20")
a.set_title("Both searches reach 0.89 by trial 4 and end level", fontsize=10)
a.grid(alpha=0.3)
a.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()
print("dots: every trial's own score (trials below 0.8 are off the plot)\n")

for s in SAMPLERS:
    final = TRIALS[s]["epoch 20"]
    curve = np.maximum.accumulate(final.to_numpy())
    print(
        f"{s:6s}: best {curve[-1]:.3f}, found at trial {final.idxmax()}; "
        f"best-so-far reaches 0.89 at trial {np.argmax(curve >= 0.89)}"
    )
same = [STORED["random"][n]["params"] == STORED["tpe"][n]["params"] for n in range(60)]
print(
    f"identical trials: 0 to {same.index(False) - 1}; the searches part at trial {same.index(False)}"
)
good = pd.concat(TRIALS.values())
lr = good.loc[good["epoch 20"] >= 0.88, "lr"]
print(
    f"all 120 trials ending at 0.88 or above: {len(lr)}, lr from {lr.min():.0e} to {lr.max():.0e}"
)

**Reading.** The lines end level: both searches find 0.894, and both pass 0.89 at trial 4,
while they were still identical (the first ten dots lie on top of each other). Nor does
TPE get there sooner: random finds its 0.894 at trial 24, TPE at trial 43. The top of this
space is flat and wide: trials that end at 0.88 or above have learning rates from 2e-5 to
7e-3, more than half of the four decades of the axis. A random draw lands there often enough
that 60 trials find the top either way.

By best score, then, TPE bought nothing here.

**If both end at 0.894, what did TPE buy?** Before you run the next cell, look at the dots
in the figure above and name one measure on which you expect TPE to beat random.

**Setup.** Random draws the lr axis evenly, so its 60 trials are a fair map of it: the band
from 1e-4 to 3e-3 is where none of them fails. The cell checks that map, then compares the two
searches on trials 10 to 59, from where they part. Compute is the sum of the trial seconds on
the machine that ran the script.

In [ ]:
BAND = (1e-4, 3e-3)


def in_band(lr: pd.Series) -> pd.Series:
    return (lr >= BAND[0]) & (lr <= BAND[1])


r = TRIALS["random"]
inside = in_band(r["lr"])
print("random's 60 trials as a map of the lr axis:")
for name, part in [("in the band", r[inside]), ("outside", r[~inside])]:
    print(
        f"  {name:12s} {len(part):2d} trials, median {part['epoch 20'].median():.3f}, "
        f"below 0.8: {(part['epoch 20'] < 0.8).sum()}"
    )

PARTED = slice(10, None)  # trials 10 to 59: from where the two searches part
table = pd.DataFrame(
    {
        s: {
            "lr in [1e-4, 3e-3]": f"{in_band(t['lr']).mean():.0%}",
            "median final accuracy of a trial": f"{t['epoch 20'].median():.3f}",
            "trials below 0.8": f"{(t['epoch 20'] < 0.8).sum()}",
            "compute (sum of trial seconds)": f"{t['seconds'].sum():.0f} s",
        }
        for s, t in ((s, TRIALS[s].iloc[PARTED]) for s in SAMPLERS)
    }
)
display(
    table.rename(columns=NAME).style.set_caption(
        "where trials 10 to 59 of each search went"
    )
)

parted = TRIALS["tpe"].iloc[PARTED]
((top, n),) = Counter(
    zip(parted["width"], parted["depth"], parted["batch"])
).most_common(1)
print(
    f"TPE's trials 10 to 59: {n} of {len(parted)} are {top[0]} x {top[1]} at batch {top[2]}"
)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for a, s in zip(axs, SAMPLERS):
    t = TRIALS[s]
    ok = t["epoch 20"] >= 0.8
    a.axhspan(*BAND, color="0.92", zorder=0)
    points = a.scatter(
        t.index[ok],
        t["lr"][ok],
        c=t["epoch 20"][ok],
        cmap="viridis",
        vmin=0.8,
        vmax=0.9,
        s=28,
    )
    a.scatter(t.index[~ok], t["lr"][~ok], marker="x", color="#CC0000", s=30)
    a.axvline(9.5, color="0.5", lw=1, ls=":")
    a.set_yscale("log")
    a.set_title(f"{NAME[s]}  (x: ends below 0.8)", fontsize=10)
    a.set_xlabel("trial number")
    a.grid(alpha=0.3)
axs[0].set_ylabel("lr (log scale); grey band: 1e-4 to 3e-3")
fig.colorbar(points, ax=axs, label="validation accuracy after epoch 20")
fig.suptitle(
    "Random draws the whole lr axis to the end; from trial 10 on TPE draws mostly inside the band"
)
plt.show()

**Reading.** On random's map, the band is where nothing fails: its 28 trials there have a
median of 0.885 and none ends below 0.8; all 11 failures lie outside (a few good trials do too,
random's winner among them). From trial 10 on, 44% of random's trials are in the band, against
82% of TPE's. So TPE's typical trial is better (median 0.885 against 0.877), and none of its
trials ends below 0.8, where 9 of random's do.

TPE also spent less than half the compute, 82 s against 189 s, but not because it looks at
cost. The trial it draws near is cheap: 41 of its 50 trials are trial 4's 256 x 2 at batch
512, which takes 27 updates per epoch where batch 64 takes 214. Had the best of the first ten
been a wide head at batch 64, TPE would have spent more.

So TPE's gain here is fewer wasted trials, not a higher best score; in a space where the good
region is narrow, random's waste would cost the best score too. Each sampler ran with one seed; another seed would
move these shares and could turn the order of the best scores around.

## 3.3.4 Successive halving: many trials short, the best ones longer

Both searches trained every trial for 20 epochs, even trials far behind after the first few.

**Successive halving** decides in rounds (the name comes from keeping half each round; with
`reduction_factor=3` it keeps a third). A **rung** is an epoch at which a trial is compared
with the other trials that reached it. The first rung is at epoch `min_resource`. At each rung
only the best part goes on, one trial in `reduction_factor`, and the next rung lies
`reduction_factor` times further. With `min_resource=1` and `reduction_factor=3` the rungs are
at epochs 1, 3 and 9; a trial that passes all three trains its 20 epochs.

<svg width="720" height="372" viewBox="0 0 720 372" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="13">
<rect width="720" height="372" fill="white"/>
<text x="10" y="18" font-weight="bold" fill="#222">Successive halving, reduction factor 3: the epochs each of 9 trials trains</text>
<rect x="90" y="50" width="27" height="14" rx="3" fill="#999999"/><text x="80" y="61" text-anchor="end" font-size="11" fill="#222">trial 1</text><rect x="90" y="72" width="27" height="14" rx="3" fill="#999999"/><text x="80" y="83" text-anchor="end" font-size="11" fill="#222">trial 2</text><rect x="90" y="94" width="27" height="14" rx="3" fill="#999999"/><text x="80" y="105" text-anchor="end" font-size="11" fill="#222">trial 3</text><rect x="90" y="116" width="27" height="14" rx="3" fill="#999999"/><text x="80" y="127" text-anchor="end" font-size="11" fill="#222">trial 4</text><rect x="90" y="138" width="27" height="14" rx="3" fill="#999999"/><text x="80" y="149" text-anchor="end" font-size="11" fill="#222">trial 5</text><rect x="90" y="160" width="27" height="14" rx="3" fill="#999999"/><text x="80" y="171" text-anchor="end" font-size="11" fill="#222">trial 6</text><rect x="90" y="182" width="81" height="14" rx="3" fill="#999999"/><text x="80" y="193" text-anchor="end" font-size="11" fill="#222">trial 7</text><rect x="90" y="204" width="81" height="14" rx="3" fill="#999999"/><text x="80" y="215" text-anchor="end" font-size="11" fill="#222">trial 8</text><rect x="90" y="226" width="540" height="14" rx="3" fill="#009E73"/><text x="80" y="237" text-anchor="end" font-size="11" fill="#222">trial 9</text>
<line x1="117" y1="40" x2="117" y2="252" stroke="#444" stroke-dasharray="4 3"/><text x="117" y="36" text-anchor="middle" font-size="11" fill="#222">rung 1</text><line x1="171" y1="40" x2="171" y2="252" stroke="#444" stroke-dasharray="4 3"/><text x="171" y="36" text-anchor="middle" font-size="11" fill="#222">rung 2</text><line x1="333" y1="40" x2="333" y2="252" stroke="#444" stroke-dasharray="4 3"/><text x="333" y="36" text-anchor="middle" font-size="11" fill="#222">rung 3</text>
<line x1="90" y1="252" x2="630" y2="252" stroke="#444"/>
<line x1="90" y1="252" x2="90" y2="257" stroke="#444"/><text x="90" y="270" text-anchor="middle" font-size="11" fill="#222">0</text><line x1="117" y1="252" x2="117" y2="257" stroke="#444"/><text x="117" y="270" text-anchor="middle" font-size="11" fill="#222">1</text><line x1="171" y1="252" x2="171" y2="257" stroke="#444"/><text x="171" y="270" text-anchor="middle" font-size="11" fill="#222">3</text><line x1="333" y1="252" x2="333" y2="257" stroke="#444"/><text x="333" y="270" text-anchor="middle" font-size="11" fill="#222">9</text><line x1="630" y1="252" x2="630" y2="257" stroke="#444"/><text x="630" y="270" text-anchor="middle" font-size="11" fill="#222">20</text>
<text x="360" y="286" text-anchor="middle" fill="#222">epochs trained</text>
<text x="123" y="302" font-size="11" fill="#222">rung 1, epoch 1: 9 arrive, the best 3 go on</text><text x="177" y="318" font-size="11" fill="#222">rung 2, epoch 3: 3 arrive, the best 1 goes on</text><text x="339" y="334" font-size="11" fill="#222">rung 3, epoch 9: a lone arrival goes on to epoch 20</text>
<text x="640" y="237" font-size="11" fill="#009E73">20 epochs</text>
<text x="10" y="360" font-size="12" fill="#222">6 · 1 + 2 · 3 + 1 · 20 = 32 epochs, against 9 · 20 = 180 when every trial trains to the end.</text>
</svg>

**The order matters.** Optuna cannot wait for all trials to reach a rung, so it compares an
arriving trial with the trials that reached the rung before it; the first few trials meet few
rivals and pass easily.

**The bet.** Halving bets that the order after one epoch is the order after twenty. When that
holds, it saves the epochs of the losers and loses nothing. A trial that starts slowly and
finishes strong breaks the bet: like a checkpoint after the first kilometre of a marathon,
the first rung picks the sprinters. The `PercentilePruner` in lesson 2's `study.py` is another
rule that makes the same bet.

## 3.3.5 Halving replayed on the stored curves

Every stored trial has its whole curve, so Optuna's own `SuccessiveHalvingPruner` can decide
without training, and the file says what each stopped trial would have reached.

**Question.** With its first rung at epoch 1, how many epochs does halving save on these two
searches, and does it keep their winners?

**Expect.** Write down: will it keep the winner of the random search? Of the TPE search? Look
back at the first ten trials: how far apart are they after epoch 1, and after epoch 20?

**Setup.** `replay` creates a study with the pruner. Its objective trains nothing: trial `n`
reports stored trial `n`'s curve epoch by epoch (`trial.report`) and asks the pruner after each
epoch (`trial.should_prune`), in the order the trials ran. Under a real pruner the stopped slow
starters would count among TPE's rest, so the pruner would also steer TPE towards fast
starters; the replay shows only what the pruner decides on these trials.

In [ ]:
def replay(
    rows: list[dict], min_resource: int, reduction_factor: int = 3
) -> np.ndarray:
    # The epochs each stored trial trains when the SuccessiveHalvingPruner decides.
    pruner = optuna.pruners.SuccessiveHalvingPruner(
        min_resource=min_resource, reduction_factor=reduction_factor
    )
    study = optuna.create_study(direction="maximize", pruner=pruner)
    trained = np.zeros(len(rows), dtype=int)

    def objective(trial: optuna.Trial) -> float:
        curve = rows[trial.number]["curve"]
        for epoch, accuracy in enumerate(curve, start=1):
            trial.report(accuracy, step=epoch)
            trained[trial.number] = epoch
            if trial.should_prune():
                raise optuna.TrialPruned()
        return curve[-1]

    study.optimize(objective, n_trials=len(rows))
    return trained


def outcome(sampler: str, trained: np.ndarray) -> dict:
    # What halving saved and whom it stopped, against the full search.
    final = CURVES[sampler][:, -1]
    kept = trained == EPOCHS
    top10 = np.argsort(-final)[:10]
    best_kept = final[kept].max()
    return {
        "saved": f"{1 - trained.sum() / (trained.size * EPOCHS):.0%}",
        "trials run to the end": int(kept.sum()),
        "winner stopped": "yes" if not kept[final.argmax()] else "no",
        "of the top 10, stopped": int((~kept[top10]).sum()),
        "winner minus best kept": f"{final.max() - best_kept:.3f} "
        f"({round((final.max() - best_kept) * STORED['valid'])} photos)",
    }


TRAINED = {s: replay(STORED[s], min_resource=1) for s in SAMPLERS}
display(
    pd.DataFrame({NAME[s]: outcome(s, TRAINED[s]) for s in SAMPLERS}).style.set_caption(
        "successive halving, first rung at epoch 1, reduction factor 3"
    )
)
for s in SAMPLERS:
    stops = Counter(TRAINED[s].tolist())
    print(
        f"{s:6s}: "
        + ", ".join(
            f"{stops[e]} trials train {e} epoch{'s' if e > 1 else ''}"
            for e in sorted(stops)
        )
    )

**Reading.** Halving saves 77% of the epochs of the random search and 85% of the TPE search:
most trials train one epoch. More than a third passes each rung (28 of random's 60 pass the
first), because a trial is compared only with the trials that reached the rung before it, so
early trials pass easily. But it stops the winner in both searches, and 7 and 10 of their 10
best trials. Little accuracy is lost: the best trial it keeps is 0.001 and 0.005 below the
winner, 2 and 8 of the 1,515 validation photos, because on this flat top many trials end near
the best.

**Who is stopped, and who is kept?** For each search, the winner it stopped beside the best
trial it kept, with every other trial's curve in grey behind them and the rungs marked.

In [ ]:
epochs = np.arange(1, EPOCHS + 1)
fig, axs = plt.subplots(1, 2, figsize=(12, 3.9), sharey=True)
for a, s in zip(axs, SAMPLERS):
    curves, trained = CURVES[s], TRAINED[s]
    final = curves[:, -1]
    winner = int(final.argmax())
    kept = np.flatnonzero(trained == EPOCHS)
    starter = int(kept[final[kept].argmax()])
    stop = trained[winner]
    for c in curves:
        a.plot(epochs, c, color="0.85", lw=0.7, zorder=0)
    a.plot(
        epochs[:stop],
        curves[winner, :stop],
        "o-",
        color="#D55E00",
        lw=2,
        label=f"trial {winner}, the winner: stopped at epoch {stop}",
    )
    a.plot(
        epochs[stop - 1 :],
        curves[winner, stop - 1 :],
        color="#D55E00",
        lw=1.5,
        ls="--",
        label="the epochs it would have trained",
    )
    a.plot(
        epochs,
        curves[starter],
        color="#009E73",
        lw=2,
        label=f"trial {starter}: kept to the end",
    )
    for rung in (1, 3, 9):
        a.axvline(rung, color="0.3", ls=":", lw=1)
    a.set_xticks([1, 3, 5, 9, 15, 20])
    a.set_ylim(0.55, 0.91)
    a.set_title(f"{NAME[s]}: rungs at epochs 1, 3 and 9", fontsize=10)
    a.set_xlabel("epoch")
    a.grid(alpha=0.3)
    a.legend(frameon=False, fontsize=8, loc="lower right")
axs[0].set_ylabel("validation accuracy")
fig.suptitle("The first rung stops a slow starter that wins and keeps a fast starter")
plt.tight_layout()
plt.show()


def describe(sampler: str, n: int) -> str:
    row = TRIALS[sampler].loc[n]
    return (
        f"{NAME[sampler]:6s} {n:3d}  {head(row):>8s}  batch {int(row['batch']):3d}  "
        f"lr {row['lr']:.1e}  epoch 1: {row['epoch 1']:.3f}  epoch 20: {row['epoch 20']:.3f}"
    )


print("kept to the end:")
for s in SAMPLERS:
    for n in np.flatnonzero(TRAINED[s] == EPOCHS):
        print("  " + describe(s, n))
print("of the top 10, stopped at the first rung:")
for s in SAMPLERS:
    top = np.argsort(-CURVES[s][:, -1])[:10]
    cut = TRIALS[s].iloc[[n for n in top if TRAINED[s][n] == 1]]
    print(
        f"  {NAME[s]:6s} {len(cut)} trials, epoch 1: {cut['epoch 1'].min():.3f} to "
        f"{cut['epoch 1'].max():.3f}, epoch 20: {cut['epoch 20'].min():.3f} to {cut['epoch 20'].max():.3f}"
    )

**Reading.** TPE's winner, trial 43, is at 0.631 after one epoch, far down the grey pack, and
ends at 0.894; the first rung stops it, and with it the rest of TPE's top 10, all one
configuration. Random's winner, trial 24, is at 0.803 after one epoch and is stopped too. The
trials kept to the end are at 0.85 or above after one epoch (trials 0 and 2 pass as the first
arrivals).

This is the pitfall of halving: it ranks trials by how far they get in the first epochs, so it
favours fast starters over slow starters that end higher. As a rule of thumb, a low learning
rate starts slowly. On this flat top that costs little accuracy; it does decide which kind of
head the search ends with.

## 3.3.6 Where to put the first rung

A later first rung (`min_resource`) gives slow starters more epochs before they are judged.

In [ ]:
MIN_RESOURCES = [1, 2, 3, 5]

rows = []
for m in MIN_RESOURCES:
    for s in SAMPLERS:
        o = outcome(s, replay(STORED[s], min_resource=m))
        keep = (
            "saved",
            "winner stopped",
            "of the top 10, stopped",
            "winner minus best kept",
        )
        rows.append({"first rung": m, "search": NAME[s], **{k: o[k] for k in keep}})
display(
    pd.DataFrame(rows)
    .set_index(["first rung", "search"])
    .style.set_caption(
        "successive halving with reduction factor 3, by the epoch of its first rung"
    )
)

**Reading.** A first rung at epoch 3 keeps random's winner and still saves 61% of the epochs.
TPE's winner, at 0.63 after one epoch, is stopped at every setting. The best kept trial stays
within 0.005 of the winner throughout: on this search the trade-off is between epochs and
*which* top trial the search ends with, not between epochs and accuracy.

## What you now know

- **Random draws, TPE remembers.** Random draws every axis afresh and keeps spending trials on
  regions it already saw fail. TPE splits the trials so far into a good tenth and the rest and
  draws where the good trials were and the rest were not: after 10 trials, all ten of its next
  trials took the good trial's batch 512.
- **TPE's gain shows in where the trials go, not in the best score.** Both searches ended at
  0.894 on this flat, wide top, but from trial 10 on TPE put 82% of its trials in the band where
  nothing fails, against random's 44%. It also spent less compute, because the head it drew
  near is cheap, not because it looks at cost.
- **Successive halving** compares trials at rungs (here epochs 1, 3 and 9) and lets the best
  part go on; with its first rung at epoch 1 it saved 77% and 85% of the epochs.
- **Its pitfall is the slow starter**, such as a trial with a low learning rate: it stopped the
  winner of both searches and TPE's ten best, one configuration, at a cost of at most 0.005 on
  this flat top. Run with a sampler, a pruner also steers the sampler towards fast starters.

## Exercises

**E1: the rung and the factor.** In 3.3.6, pass `reduction_factor=2` and `4` to `replay`
and try first rungs up to 9. Does a setting that saves more than 61% keep random's winner? Is
there one that keeps TPE's winner and still saves epochs?

**E2: when does TPE settle?** For each block of 10 trials (0 to 9, 10 to 19, ...), compute
the share in the band 1e-4 to 3e-3 for both searches. From which block on is TPE's share
clearly above random's? Should random's share drift?

**E3: what a grid would cost.** A grid search tries every combination of a list of values per
axis. With 3 values per axis (`depth` has only 2), how many trials is that, and how many
seconds at the median of `TRIALS["random"]["seconds"]`? How many learning rates does it try, against random's 60?

**E4: read the bet.** Plot the accuracy after epoch 1 against epoch 20 for all 120 trials
(`CURVES`), coloured by `batch`. Where are the trials that halving stops wrongly? Does the order
of the top trials look settled at epoch 3, or at epoch 5?